# Imports

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import SparseLinear
from DenseMLP import DenseMLP
from SparseMLP import SparseMLP


# Loading Datasets

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256
)

100.0%
100.0%
100.0%
100.0%


# Evaluation Functions

In [ ]:
def evaluate(model):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            preds = logits.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

# Train Function

In [ ]:
def train_model(model, epochs=5, lr=1e-4):

    model = model.to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.1,
        momentum=0.9
    )

    criterion = nn.CrossEntropyLoss()

    start_time = time.time()

    for epoch in range(epochs):

        model.train()

        total_loss = 0

        for x, y in train_loader:

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x)

            loss = criterion(logits, y)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

            total_loss += loss.item()

        acc = evaluate(model)

        print(
            f"Epoch {epoch+1} | "
            f"Loss: {total_loss / len(train_loader):.4f} | "
            f"Accuracy: {acc:.4f}"
        )

    total_time = time.time() - start_time

    return acc, total_time

# Train Dense Model

In [ ]:
dense_model = DenseMLP()

dense_acc, dense_time = train_model(
    dense_model,
    epochs=5
)

print("\nDense Results")
print("Accuracy:", dense_acc)
print("Time:", dense_time)

Epoch 1 | Loss: 0.3320 | Accuracy: 0.9556
Epoch 2 | Loss: 0.1033 | Accuracy: 0.9712
Epoch 3 | Loss: 0.0712 | Accuracy: 0.9710
Epoch 4 | Loss: 0.0552 | Accuracy: 0.9764
Epoch 5 | Loss: 0.0410 | Accuracy: 0.9771

Dense Results
Accuracy: 0.9771
Time: 6.4670867919921875


# Train Sparse Model

In [ ]:
sparse_model = SparseMLP(
    keep_ratio=1
)

sparse_acc, sparse_time = train_model(
    sparse_model,
    epochs=5
)

print("\nSparse Results: 100% Keep Ratio")
print("Accuracy:", sparse_acc)
print("Time:", sparse_time)

Epoch 1 | Loss: nan | Accuracy: 0.0980
Epoch 2 | Loss: nan | Accuracy: 0.0980
Epoch 3 | Loss: nan | Accuracy: 0.0980
Epoch 4 | Loss: nan | Accuracy: 0.0980
Epoch 5 | Loss: nan | Accuracy: 0.0980

Sparse Results: 100% Keep Ratio
Accuracy: 0.098
Time: 7.527846813201904


In [ ]:
sparse_model = SparseMLP(
    keep_ratio=0.5
)

sparse_acc, sparse_time = train_model(
    sparse_model,
    epochs=5
)

print("\nSparse Results: 50% Keep Ratio")
print("Accuracy:", sparse_acc)
print("Time:", sparse_time)

Epoch 1 | Loss: nan | Accuracy: 0.0980
Epoch 2 | Loss: nan | Accuracy: 0.0980
Epoch 3 | Loss: nan | Accuracy: 0.0980
Epoch 4 | Loss: nan | Accuracy: 0.0980
Epoch 5 | Loss: nan | Accuracy: 0.0980

Sparse Results: 50% Keep Ratio
Accuracy: 0.098
Time: 7.241635799407959


In [ ]:
sparse_model = SparseMLP(
    keep_ratio=0.2
)

sparse_acc, sparse_time = train_model(
    sparse_model,
    epochs=5
)

print("\nSparse Results: 50% Keep Ratio")
print("Accuracy:", sparse_acc)
print("Time:", sparse_time)

Epoch 1 | Loss: nan | Accuracy: 0.0980
Epoch 2 | Loss: nan | Accuracy: 0.0980
Epoch 3 | Loss: nan | Accuracy: 0.0980
Epoch 4 | Loss: nan | Accuracy: 0.0980
Epoch 5 | Loss: nan | Accuracy: 0.0980

Sparse Results: 50% Keep Ratio
Accuracy: 0.098
Time: 7.211780786514282
